In [37]:
import torch

In [38]:
import torch.nn as nn
class PINN(nn.Module):
    def __init__(self, d, hidden_size=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
    def forward(self, x):
        return self.net(x)

x = torch.tensor([
    [0.0, 1.0, 2.0],
    [1.0, 2.0, 0.0],
    [2.0, 0.0, 1.0],
    [1.0, 1.0, 1.0],
    [-1.0, -1.0, -1.0],
], requires_grad=True)
bs, d = x.shape
bs, d

(5, 3)

In [39]:
model = PINN(d, hidden_size=16)
model.net

Sequential(
  (0): Linear(in_features=3, out_features=16, bias=True)
  (1): Tanh()
  (2): Linear(in_features=16, out_features=16, bias=True)
  (3): Tanh()
  (4): Linear(in_features=16, out_features=1, bias=True)
)

In [40]:
y = model(x)
y

tensor([[ 0.4600],
        [ 0.2912],
        [ 0.2566],
        [ 0.3400],
        [-0.1198]], grad_fn=<AddmmBackward0>)

In [41]:
g = torch.autograd.grad(
    inputs=x,
    outputs=y,
    grad_outputs=torch.ones_like(y),
    create_graph=True,
    retain_graph=True,
)[0]
g

tensor([[-0.0180,  0.0741,  0.0675],
        [-0.0022,  0.0664,  0.1384],
        [ 0.0082,  0.0508,  0.0699],
        [-0.0319,  0.0844,  0.1048],
        [-0.0111,  0.0568,  0.1146]], grad_fn=<MmBackward0>)

In [42]:
l0 = torch.autograd.grad(
    inputs=x,
    outputs=g[:,0],
    grad_outputs=torch.ones_like(g[:,0]),
    retain_graph=True,
    create_graph=False,
)[0]
l0

tensor([[-0.0434,  0.0155, -0.0006],
        [-0.0172,  0.0116, -0.0207],
        [ 0.0215, -0.0184, -0.0258],
        [ 0.0021, -0.0065, -0.0267],
        [ 0.0274, -0.0129,  0.0136]])

In [43]:
l1 = torch.autograd.grad(
    inputs=x,
    outputs=g[:,1],
    grad_outputs=torch.ones_like(g[:,1]),
    retain_graph=True,
    create_graph=False,
)[0]
l1

tensor([[ 0.0155, -0.0282, -0.0044],
        [ 0.0116, -0.0321, -0.0083],
        [-0.0184,  0.0115,  0.0057],
        [-0.0065, -0.0024,  0.0020],
        [-0.0129,  0.0428,  0.0082]])

In [44]:
l2 = torch.autograd.grad(
    inputs=x,
    outputs=g[:,2],
    grad_outputs=torch.ones_like(g[:,2]),
    retain_graph=False,
    create_graph=False,
)[0]
l2

tensor([[-0.0006, -0.0044, -0.0466],
        [-0.0207, -0.0083, -0.0087],
        [-0.0258,  0.0057, -0.0426],
        [-0.0267,  0.0020, -0.0474],
        [ 0.0136,  0.0082,  0.0538]])

In [60]:
# laplace
# d2f/dx2, ...
#l0[:,0]
#l1[:,1]
#l2[:,2]

In [61]:
# mixed derivatives
# d2f/dxdy = d2f/dydx
assert torch.isclose( l0[:,1], l1[:,0] ).all().item()
assert torch.isclose( l0[:,2], l2[:,0] ).all().item()
assert torch.isclose( l1[:,0], l0[:,1] ).all().item()
assert torch.isclose( l1[:,2], l2[:,1] ).all().item()
assert torch.isclose( l2[:,0], l0[:,2] ).all().item()
assert torch.isclose( l2[:,1], l1[:,2] ).all().item()